# Synthetic performance evaluation

How well does a COCO-pretrained instance segmentation model find resistor bodies when it is
trained on the small synthetic set, and how much does task-specific tuning add?

Three stages, all scored the same way:

| Stage | Model | Purpose |
|---|---|---|
| 0. Zero-shot | YOLOv8m-seg, COCO weights, no training | Floor: COCO has no resistor class |
| 1. Baseline | YOLOv8m-seg, COCO backbone, fine-tuned with ultralytics' default settings | What the pretrained backbone gives out of the box |
| 2. Tuned | Same start, `configs/train.toml` augmentation, cosine LR, confidence threshold picked on val | What tuning adds |

**Data.** `synth` (100 frames) is train. `synth_val` and `synth_test` (20 frames each) are new sets
from the same generator with other seeds, so test frames are unseen but **from the same
distribution**. These numbers are therefore an upper bound on what to expect on real frames; the
real test sessions in `docs/plan.md` are what count.

**Scoring.** Two levels:

- *Tile level*: ultralytics mask mAP50-95 / mAP50 on the 1024 px test tiles (comparable to the
  fibre project's `AP_mask`).
- *Frame level*: the robot's view. Full 4608 × 2592 test frames are tiled, predicted and merged
  with `robot_offline.tiling`, then `robot_offline.evaluate` matches predictions to labels
  (IoU ≥ 0.5) and reports centre, yaw, length and width error in mm and degrees, misses, false
  detections and merges, against the `docs/plan.md` targets (centre < 1 mm **and** yaw < 3° for
  95 % of bodies).

Training cells skip themselves when the run's `best.pt` already exists, so the notebook can be
re-run for the analysis alone. Delete `runs/synth_eval/<stage>/` to retrain a stage.

In [ ]:
import json
import os
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from shapely.geometry import Polygon
from ultralytics import YOLO

from robot_offline import config, evaluate, export_yolo, manifest, tiling, train, visualise

ROOT = config.ROOT
os.chdir(ROOT)                       # ultralytics drops helper downloads in the working folder
pd.set_option("display.precision", 3)

DS = config.load("dataset")
PX_PER_MM = DS["scale"]["px_per_mm"]
SPLITS = {"synth": "train", "synth_val": "val", "synth_test": "test"}
YOLO_DIR = config.DATA_DIR / "yolo" / "synth_eval"
RUNS = ROOT / "runs" / "synth_eval"

EPOCHS = 60                          # same for both trained stages, so only the settings differ
BATCH = 4                            # fits YOLOv8m-seg at 1024 px in 6 GB (4.7 GB used)
WORKERS = 2                          # more dataloader workers crash on Windows (shared memory)
START_WEIGHTS = "yolov8m-seg.pt"     # configs/train.toml [model] weights

print(f"torch {torch.__version__}, CUDA {torch.cuda.is_available()}, "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1. Data

In [ ]:
labels = manifest.load(DS["labels"]["rules_version"])
samples = {name: [s for s in labels["samples"] if s["session"] == name] for name in SPLITS}
rows = []
for name, split in SPLITS.items():
    n = [len(s["instances"]) for s in samples[name]]
    rows.append({"set": name, "split": split, "frames": len(n), "bodies": sum(n),
                 "empty frames": sum(k == 0 for k in n), "bodies/frame": np.mean(n)})
pd.DataFrame(rows).set_index("set")

In [ ]:
def show_frame(sample, predictions=None, title=None, ax=None):
    # Labels green; predictions magenta when given. Cropped to the work area.
    image = cv2.imread(str(config.DATA_DIR / sample["image"]))
    image = visualise.draw_polygons(image, manifest.polygons(sample), (0, 200, 0), 4)
    if predictions is not None:
        image = visualise.draw_polygons(image, predictions, (255, 0, 255), 4)
    a = DS["work_area"]
    image = image[a["y0"]:a["y0"] + a["height"], a["x0"]:a["x0"] + a["width"]]
    ax = ax or plt.figure(figsize=(14, 7)).gca()
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title or sample["image"])
    ax.axis("off")

show_frame(samples["synth_test"][0], title="A test frame, work area, labels in green")

### Tiles

The work area (3600 × 1800 px) is cut into 8 overlapping 1024 px tiles per frame at full
resolution, the same tiling the robot will use at inference. A body is labelled in a tile if at
least 60 % of it is inside (`[tiles]` in `configs/dataset.toml`). Tiles with no bodies are kept so
the model learns empty table.

In [ ]:
data_yaml = export_yolo.export_from_config(labels["samples"], SPLITS, YOLO_DIR)
data_yaml

## 2. Scoring helpers

`score(model, conf)` runs a model over the full test frames with tiling and returns the frame-level
summary plus the per-frame results. `tile_map(weights)` is ultralytics' own mask mAP on the test
tiles.

In [ ]:
def score(model, frames=None, conf=0.25, **kw):
    frames = frames if frames is not None else samples["synth_test"]
    results = evaluate.evaluate_samples(model, frames, conf=conf, **kw)
    summary = evaluate.summarise(results)
    summary["s_per_frame"] = float(np.median([r["seconds"] for r in results]))
    return summary, results


def tile_map(weights, split="test"):
    m = YOLO(str(weights)).val(data=str(data_yaml), split=split, imgsz=DS["tiles"]["size"],
                               batch=BATCH, workers=WORKERS, plots=False, verbose=False)
    return {"tile_mask_mAP50-95": m.seg.map, "tile_mask_mAP50": m.seg.map50,
            "tile_box_mAP50-95": m.box.map}


def body_table(results):
    return pd.DataFrame([m for r in results for m in r["matched"]])


SHOW = ["recall", "precision", "f1", "misses", "false_detections", "false_det_on_empty", "merges",
        "best_iou_mean", "centre_mm_p50", "centre_mm_p95", "yaw_deg_p50", "yaw_deg_p95",
        "length_mm_bias", "width_mm_bias", "width_mm_p95_abs", "on_target_fraction",
        "meets_centre_yaw_target", "s_per_frame"]
report = {}

## 3. Stage 0: zero-shot COCO weights

COCO has no resistor class, so this is the floor. Every COCO class is accepted (class-agnostic),
at a low confidence of 0.05 to give it every chance.

In [ ]:
coco = YOLO(str(train.pretrained(START_WEIGHTS)))
zs_summary, zs_results = score(coco, conf=0.05, agnostic_nms=True)
zs_classes = pd.Series([coco.names[int(c)] for r in coco.predict(
    [cv2.imread(str(config.DATA_DIR / s["image"])) for s in samples["synth_test"][:3]],
    conf=0.05, verbose=False) for c in r.boxes.cls]).value_counts()
report["0 zero-shot"] = {**zs_summary, "tile_mask_mAP50-95": 0.0, "tile_mask_mAP50": 0.0}
print("COCO classes predicted on 3 whole test frames:", dict(zs_classes) or "none")
pd.Series(zs_summary)[SHOW].to_frame("zero-shot")

## 4. Stage 1: baseline fine-tune

COCO-pretrained YOLOv8m-seg, every layer trainable, ultralytics' default hyperparameters and
augmentation (no rotation, no vertical flip, ±50 % scale, strong colour jitter). Only the image
size, batch and epochs are set.

In [ ]:
def train_stage(name, augment, **overrides):
    best = RUNS / name / "weights" / "best.pt"
    if best.exists():
        print(f"{name}: using existing {best}")
    else:
        t0 = time.perf_counter()
        train.train(data_yaml, weights=START_WEIGHTS, augment=augment, project="runs/synth_eval",
                    name=name, epochs=EPOCHS, batch=BATCH, workers=WORKERS, patience=EPOCHS, plots=True, **overrides)
        print(f"{name}: trained in {(time.perf_counter() - t0) / 60:.0f} min")
    return best


baseline_weights = train_stage("baseline", augment=False)

In [ ]:
def plot_curves(names):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for name in names:
        log = pd.read_csv(RUNS / name / "results.csv")
        log.columns = log.columns.str.strip()
        axes[0].plot(log["epoch"], log["train/seg_loss"], label=f"{name} train")
        axes[0].plot(log["epoch"], log["val/seg_loss"], "--", label=f"{name} val")
        axes[1].plot(log["epoch"], log["metrics/mAP50-95(M)"], label=name)
        axes[2].plot(log["epoch"], log["metrics/mAP50(M)"], label=name)
    for ax, t in zip(axes, ["mask loss", "val mask mAP50-95", "val mask mAP50"]):
        ax.set_title(t); ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()

plot_curves(["baseline"])

In [ ]:
baseline = YOLO(str(baseline_weights))
bl_summary, bl_results = score(baseline)
report["1 baseline"] = {**bl_summary, **tile_map(baseline_weights)}
pd.Series(report["1 baseline"])[SHOW + ["tile_mask_mAP50-95", "tile_mask_mAP50"]].to_frame("baseline")

## 5. Stage 2: tuned

Changes from the baseline, each for a reason specific to this camera and task:

| Setting | Baseline | Tuned | Why |
|---|---|---|---|
| `degrees` | 0 | 180 | Resistors lie at any yaw; the overhead view has no up |
| `flipud` | 0 | 0.5 | Same: vertical flips are as realistic as horizontal |
| `scale` | 0.5 | 0.1 | Fixed camera height: body size in px barely changes |
| `hsv_h` / `hsv_s` | 0.015 / 0.7 | 0.01 / 0.3 | Keep band colours recognisable for the later classifier crop |
| `cos_lr` | off | on | Smoother end of the schedule on a small set |
| confidence | 0.25 | picked on `synth_val` | Best F1 on val, never on test |

The first four come from `configs/train.toml [augment]`.

In [ ]:
tuned_weights = train_stage("tuned", augment=True, cos_lr=True)
plot_curves(["baseline", "tuned"])

### Confidence threshold, chosen on val

The threshold trades misses against false detections. It is picked on `synth_val` by F1 and then
applied once to test.

In [ ]:
tuned = YOLO(str(tuned_weights))
sweep = []
for conf in [0.1, 0.25, 0.4, 0.5, 0.6, 0.7, 0.8]:
    s, _ = score(tuned, frames=samples["synth_val"], conf=conf)
    sweep.append({"conf": conf, **{k: s[k] for k in ["recall", "precision", "f1", "misses",
                                                     "false_detections", "on_target_fraction"]}})
sweep = pd.DataFrame(sweep).set_index("conf")
best_conf = float(sweep["f1"].idxmax())
print(f"chosen confidence: {best_conf}")
sweep

In [ ]:
tn_summary, tn_results = score(tuned, conf=best_conf)
report["2 tuned"] = {**tn_summary, **tile_map(tuned_weights), "conf": best_conf}
pd.Series(report["2 tuned"])[SHOW + ["tile_mask_mAP50-95", "tile_mask_mAP50"]].to_frame("tuned")

## 6. Comparison

In [ ]:
cols = ["tile_mask_mAP50-95", "tile_mask_mAP50"] + SHOW
table = pd.DataFrame(report).T[cols]
table.T

In [ ]:
bodies = {"baseline": body_table(bl_results), "tuned": body_table(tn_results)}
specs = [("centre_mm", "centre error (mm)", evaluate.CENTRE_TARGET_MM),
         ("yaw_deg", "yaw error (deg)", evaluate.YAW_TARGET_DEG),
         ("length_mm", "length error (mm, signed)", None),
         ("width_mm", "width error (mm, signed)", None)]
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, (col, title, target) in zip(axes, specs):
    for name, df in bodies.items():
        ax.hist(df[col], bins=40, alpha=0.55, label=name)
    if target:
        ax.axvline(target, color="k", ls="--", lw=1, label="target")
    ax.set_title(title); ax.legend()
plt.tight_layout()

### Where the errors are

Length error split by package size (¼ W bodies are about 6 mm long, ½ W about 9 mm), and the
mean centre offset in x and y, which would show a systematic shift between predicted and labelled
outlines.

In [ ]:
df = bodies["tuned"].copy()
df["package"] = np.where(df["true_length_mm"] < 7.5, "1/4 W", "1/2 W")
print("mean centre offset (mm): dx = %.3f, dy = %.3f" % (df["dx_mm"].mean(), df["dy_mm"].mean()))
df.groupby("package")[["iou", "centre_mm", "yaw_deg", "length_mm", "width_mm"]].agg(["mean", "std"])

### Worst test bodies (tuned), and a full test frame

In [ ]:
def worst_crops(results, key="centre_mm", n=8, half=60, scale=4):
    found = []
    for r, s in zip(results, samples["synth_test"]):
        gts = manifest.polygons(s)
        for g in gts:
            best = max(r["polygons"], key=lambda p: Polygon(p).intersection(Polygon(g)).area, default=None)
            if best is None:
                continue
            m = evaluate.score_frame([g], [best], PX_PER_MM)["matched"]
            err = m[0][key] if m else np.inf
            found.append((err, s, g, best))
    found.sort(key=lambda f: -f[0])
    fig, axes = plt.subplots(1, n, figsize=(2.4 * n, 2.8))
    for ax, (err, s, g, p) in zip(axes, found[:n]):
        img = cv2.imread(str(config.DATA_DIR / s["image"]))
        c = g.mean(axis=0).astype(int)
        crop = img[c[1] - half:c[1] + half, c[0] - half:c[0] + half]
        crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_NEAREST)
        off = c - half
        for poly, colour in [(g, (0, 200, 0)), (p, (255, 0, 255))]:
            pts = np.round(((poly - off) * scale + scale / 2) * 16).astype(np.int32)
            cv2.polylines(crop, [pts], True, colour, 1, cv2.LINE_AA, shift=4)
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax.axis("off")
        ax.set_title(f"{key} {err:.2f}" if np.isfinite(err) else "unmatched", fontsize=9)
    plt.tight_layout()

worst_crops(tn_results, "centre_mm")
worst_crops(tn_results, "yaw_deg")

In [ ]:
r = max(tn_results, key=lambda r: r["labels"])
show_frame(samples["synth_test"][tn_results.index(r)], r["polygons"],
           title=f"Tuned on the busiest test frame: labels green, predictions magenta "
                 f"({r['labels']} bodies, {r['predictions']} predicted)")

## 7. Findings

_Written after the run; see the cell below for the numbers it is based on._

In [ ]:
table.T.to_json(RUNS / "report.json", indent=2, default_handler=str)
print(f"saved {RUNS / 'report.json'}")